# Loop contributions in the shallow-lake model

## 1. Setup: model, bifurcation diagram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from scipy.integrate import odeint
import warnings

from tigramite import data_processing as pp
from tigramite.graphs import Graphs
from tigramite.plotting import plot_time_series_graph
from tigramite.models import LinearMediation

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


colors = px.colors.qualitative.Prism

C = {
    'A_total':    colors[3],
    'A_direct':   colors[4],
    'A_nutrient': colors[8],
    'A_trophic':  colors[2],
    'A_mixed':  colors[10],
    'clear':    colors[6],
    'turbid':   colors[2],
    'L':        colors[7],
    'F':        colors[0],
    'A':        colors[4],
    'M':        colors[6],
    'Z':        colors[2],
    'N':        colors[8],
    'PF':       colors[1],
}

In [ ]:
# ===== Parameters used in the model =====
params = dict(
    # Nutrients
    flush=0.03, c_PA=0.6, c_PM=0.6, eps_N=1e-2,

    # Algae
    g_A=0.45, K_A=5.0, m_A=0.12, c_A=0.021,
    a=0.21, h_A=1.0, eps_A=1e-2,

    # Macrophytes
    g_M=0.375, K_M=1.5, s=3.0,
    m_M=0.03, c_M=0.021, eps_M=1e-2,

    # Zooplankton
    e_Z=0.5, m_Z=0.006, c_Z=0.024, eps_Z=1e-2,

    # Planktivorous fish
    p_F=0.075, e_F=0.7,
    m_F=0.0045, c_F=0.036, eps_PF=1e-2,

    # Macrophyte refuge
    q_ref=2.0,
)


# 7-variable state: L is exogenous nutrient loading, F is exogenous fishing pressure.
state_names = ['L', 'N', 'A', 'M', 'Z', 'PF', 'F']
iL, iN, iA, iM, iZ, iPF, iF = 0, 1, 2, 3, 4, 5, 6

# Local indices for the 5x5 Jacobian / covariance matrices (N,A,M,Z,PF = 0..4)
reduced_state_names = ['N', 'A', 'M', 'Z', 'PF']
jN, jA, jM, jZ, jPF = 0, 1, 2, 3, 4

FAST, SLOW = 1, 2
LAGS = {
    'l_L_N':   0,                     # exogenous
    'l_N_N':   FAST, 'l_M_N': FAST, 'l_A_N': FAST,   # -> N (fast)
    'l_N_A':   FAST, 'l_A_A': FAST, 'l_Z_A': FAST,   # -> A (fast)
    'l_N_M':   SLOW, 'l_A_M': SLOW, 'l_M_M': FAST,   # -> M (slow)
    'l_A_Z':   FAST, 'l_Z_Z': FAST, 'l_PF_Z': FAST, 'l_M_Z':   FAST, # -> Z (fast)
    'l_Z_PF':  SLOW, 'l_PF_PF': FAST, 'l_M_PF':  SLOW,  # -> PF (slow)
    'l_F_PF':  0,                     # exogenous
}

In [ ]:
def model_without_delay(state, L, F, p=params):
    """Model without delay (instantaneous stochastic differential equations),
    5 state variables: N, A, M, Z, PF; L, F are exogenous. Used in branches_by_continuation
    to relax to equilibria (equilibria and folds do not depend on the delays)."""
    N, A, M, Z, PF = state
    GA  = p['g_A'] * N/(p['K_A'] + N) * A
    GM  = p['g_M'] * N/(p['K_M'] + N) * M / (1.0 + p['s'] * A*A)
    fr  = A / (A + p['h_A'])
    graze = p['a'] * fr * Z
    gain  = p['e_Z'] * p['a'] * fr * Z
    muA = p['m_A']*A + p['c_A']*A*A
    muM = p['m_M']*M + p['c_M']*M*M
    pf_eff = p['p_F'] / (1.0 + p['q_ref']*M)
    dN  = L - p['flush']*N - GA - GM + p['c_PA']*muA + p['c_PM']*muM + p['eps_N']
    dA  = p['eps_A'] + GA    - graze        - muA
    dM  = p['eps_M'] + GM                   - muM
    dZ  = p['eps_Z'] + gain  - pf_eff*Z*PF  - p['m_Z']*Z  - p['c_Z']*Z*Z
    dPF = p['eps_PF']+ p['e_F']*pf_eff*Z*PF - F*PF - p['m_F']*PF - p['c_F']*PF*PF
    return np.array([dN, dA, dM, dZ, dPF])

In [ ]:
def simulate_sdde(state0, p, T, dt=0.1, L_mean=1.0, F_mean=0.0,
                  noise_std=0.01, sigma_L=0, sigma_F=0,
                  seed=0, lags=LAGS, additive_noise=False):
    """Euler-Maruyama simulation of the 7-variable SDDE (each link read with its own lag in `lags`).
    Multiplicative noise unless additive_noise. Returns traj of shape (T, 7) = [L,N,A,M,Z,PF,F], one row per dt."""
    rng = np.random.default_rng(seed)
    s0 = np.asarray(state0, float); s0_5 = s0[1:6] if len(s0) >= 7 else s0
    L0 = float(np.mean(L_mean)); F0 = float(np.mean(F_mean))
    lags_dt = {k:int(round(v/dt)) for k,v in lags.items()}

    rl = {k:max(v-1,0) for k,v in lags_dt.items()}
    max_lag = max(lags_dt.values())
    hist = np.tile(np.array([L0,*s0_5,F0],float),(max_lag+1,1))
    traj = np.zeros((T,7)); state = s0_5.copy(); sigma = np.full(5,noise_std)
    L_n_arr = (np.full(T,L_mean,float) if np.isscalar(L_mean) else np.asarray(L_mean,float)) + sigma_L*rng.standard_normal(T)
    F_n_arr = (np.full(T,F_mean,float) if np.isscalar(F_mean) else np.asarray(F_mean,float)) + sigma_F*rng.standard_normal(T)

    for n in range(T):
        N, A, M, Z, PF = state
        L_n, F_n = L_n_arr[n], F_n_arr[n]
        traj[n] = [L_n, N, A, M, Z, PF, F_n]
        # Cross-couplings at their (off-by-one-corrected) link lags:
        N_for_A = hist[rl['l_N_A'], iN]
        N_for_M = hist[rl['l_N_M'], iN]         
        A_for_M = hist[rl['l_A_M'], iA]         
        Z_for_A = hist[rl['l_Z_A'], iZ]         
        A_for_Z = hist[rl['l_A_Z'], iA]         
        M_for_N = hist[rl['l_M_N'], iM]         
        PF_for_Z= hist[rl['l_PF_Z'], iPF]       
        Z_for_PF= hist[rl['l_Z_PF'], iZ]        
        M_for_Z = hist[rl['l_M_Z'], iM]         
        M_for_PF= hist[rl['l_M_PF'], iM]        
        N_for_N = hist[rl['l_N_N'], iN]         
        A_for_N = hist[rl['l_A_N'], iA]         
        # Self-loops are all lag 1
        # only the exogenous drivers L, F act at lag 0.
        A_self  = hist[rl['l_A_A'],   iA]
        M_self  = hist[rl['l_M_M'],   iM]
        Z_self  = hist[rl['l_Z_Z'],   iZ]
        PF_self = hist[rl['l_PF_PF'], iPF]
        # dN
        GA_s = p['g_A']*N_for_N/(p['K_A']+N_for_N)*A_for_N
        GM_s = p['g_M']*N_for_N/(p['K_M']+N_for_N)*M_for_N/(1.0+p['s']*A_for_N**2)
        muA_N = p['m_A']*A_for_N + p['c_A']*A_for_N**2
        muM_N = p['m_M']*M_for_N + p['c_M']*M_for_N**2
        dN = L_n - p['flush']*N_for_N - GA_s - GM_s + p['c_PA']*muA_N + p['c_PM']*muM_N + p['eps_N']
        # dA
        GA = p['g_A']*N_for_A/(p['K_A']+N_for_A)*A_self
        graze = p['a']*A_self/(A_self+p['h_A'])*Z_for_A
        muA = p['m_A']*A_self + p['c_A']*A_self**2
        dA = p['eps_A']+ GA - graze - muA
        # dM
        GM = p['g_M']*N_for_M/(p['K_M']+N_for_M)*M_self/(1.0+p['s']*A_for_M**2)
        muM = p['m_M']*M_self + p['c_M']*M_self**2
        dM = p['eps_M']+ GM - muM
        # dZ
        gain = p['e_Z']*p['a']*A_for_Z/(A_for_Z+p['h_A'])*Z_self
        pf_eff_Z = p['p_F']/(1.0+p['q_ref']*M_for_Z)
        dZ = p['eps_Z'] + gain - pf_eff_Z*Z_self*PF_for_Z - p['m_Z']*Z_self - p['c_Z']*Z_self**2
        # dPF
        pf_eff_PF = p['p_F']/(1.0+p['q_ref']*M_for_PF)
        dPF = p['eps_PF']+ p['e_F']*pf_eff_PF*Z_for_PF*PF_self - F_n*PF_self - p['m_F']*PF_self - p['c_F']*PF_self**2
        drift=np.array([dN,dA,dM,dZ,dPF]); eta=rng.standard_normal(5)

        incr = sigma*np.sqrt(dt)*eta if additive_noise else sigma*state*np.sqrt(dt)*eta
        new=state+dt*drift+incr
        hist[1:]=hist[:-1]; hist[0]=[L_n,*new,F_n]; state=new
    return traj

### Equilibria and bifurcation diagram in L

In [ ]:
def find_equilibria_from_initial_conditions(L, F, p, T_integrate=8000, dt=0.1):
    """Relax two initial conditions under the noise-free SDDE to their respective
    equilibria. Returns two 7-element arrays [L, N, A, M, Z, PF, F]."""
    x0_clear  = np.array([L, 2.0, 0.1, 8.0, 0.5, 0.4, F])
    x0_turbid = np.array([L, 5.0, 5.0, 0.1, 1.0, 0.5, F])
    n = int(T_integrate / dt)
    sol_c = simulate_sdde(x0_clear,  p, n, dt=dt, L_mean=L, F_mean=F,
                          noise_std=0.0, sigma_L=0, sigma_F=0)[n//2:].mean(0)
    sol_t = simulate_sdde(x0_turbid, p, n, dt=dt, L_mean=L, F_mean=F,
                          noise_std=0.0, sigma_L=0, sigma_F=0)[n//2:].mean(0)
    return sol_c, sol_t

In [ ]:
def branches_by_continuation(scan_values, which, fixed, params,
                             t_relax=5000, n_t=400):
    """Trace clear & turbid branches by continuation, seeding each step from the
    previous equilibrium. `which`='L' scans L at fixed F (=fixed); `which`='F' scans
    F at fixed L (=fixed). Undelayed dynamics (equilibria & folds are delay-independent);
    continuation, not a fixed initial condition, is what follows each branch to its true
    fold. Returns (clear, turbid) as (len(scan), 7) arrays [L,N,A,M,Z,PF,F], matching
    find_equilibria_from_initial_conditions' layout so iA/iM indexing is unchanged."""
    sv = np.asarray(scan_values, float)
    t = np.linspace(0, t_relax, n_t)

    def LF(v):
        return (v, fixed) if which == 'L' else (fixed, v)

    def relax(s5, v):
        L, F = LF(v)
        out = odeint(lambda x, tt: model_without_delay(np.array(x), L=L, F=F, p=params),
                     s5, t, mxstep=10000)[-1]
        return out, L, F

    L0, F0 = LF(sv[0]); L1, F1 = LF(sv[-1])
    clear = np.zeros((len(sv), 7))
    s = find_equilibria_from_initial_conditions(L=L0, F=F0, p=params)[0][1:6]
    for i, v in enumerate(sv):                       # forward continuation (clear)
        s, L, F = relax(s, v); clear[i] = [L, *s, F]
    turbid = np.zeros((len(sv), 7))
    s = find_equilibria_from_initial_conditions(L=L1, F=F1, p=params)[1][1:6]
    for i, v in reversed(list(enumerate(sv))):       # backward continuation (turbid)
        s, L, F = relax(s, v); turbid[i] = [L, *s, F]
    return clear, turbid

**Bifurcation diagram: scan L at fixed F=0.01**

In [ ]:
L_scan = np.linspace(0.0, 1.7, 80)
F_exogenous = 0.01
clear_branch, turbid_branch = branches_by_continuation(L_scan, 'L', F_exogenous, params)

fig_bif_diagram = make_subplots(rows=1, cols=2, shared_xaxes=False, horizontal_spacing=0.15)
arrow_step = 8
for col, var_idx in enumerate([iA, iM], start=1):
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan, y=clear_branch[:, var_idx],
        mode='lines', name='from clear state', line=dict(color=colors[6], dash='dashdot'),
        legendgroup='clear', showlegend=(col == 1)), row=1, col=col)
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan[5::arrow_step], y=clear_branch[5::arrow_step, var_idx],
        mode='markers', marker=dict(symbol='arrow', color=colors[6], size=10, angleref='previous'),
        legendgroup='clear', showlegend=False), row=1, col=col)
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan, y=turbid_branch[:, var_idx],
        mode='lines', name='from turbid state', line=dict(color=colors[2], dash='dot'),
        legendgroup='turbid', showlegend=(col == 1)), row=1, col=col)
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan[5::arrow_step], y=turbid_branch[5::arrow_step, var_idx],
        mode='markers', marker=dict(symbol='arrow', color=colors[2], size=10, angleref='previous', angle=180),
        legendgroup='turbid', showlegend=False), row=1, col=col)
fig_bif_diagram.update_xaxes(title_text='Nutrient loading L')
fig_bif_diagram.update_yaxes(title_text='Algae A', row=1, col=1)
fig_bif_diagram.update_yaxes(title_text='Macrophytes M', row=1, col=2)
fig_bif_diagram.update_layout(height=400, width=800, template='simple_white',
    legend=dict(x=0.22, y=1.15, orientation='h', entrywidth=130))
fig_bif_diagram.show()

**F_exogenous scan for bifurcation overlay**

In [ ]:
F_values = [-0.0, 0.01, 0.1]
F_colors = [colors[3], colors[5], colors[7]]
F_dashes = ['dashdot', 'dash', 'dot']
L_scan_F = np.linspace(0.0, 1.7, 80)
arrow_steps = [19, 22, 20]
fig_effect_F_bif_diagram = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
for f_idx, (F_val, col_hex, dash) in enumerate(zip(F_values, F_colors, F_dashes)):
    cb, tb = branches_by_continuation(L_scan_F, 'L', F_val, params)
    sizes = [12 if (i - f_idx) % arrow_steps[f_idx] == 0 else 0 for i in range(len(L_scan_F))]
    fig_effect_F_bif_diagram.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col_hex, dash=dash, width=2), name=f'F = {F_val}'))
    for var_col, var_idx in [(1, iA), (2, iM)]:
        fig_effect_F_bif_diagram.add_trace(go.Scatter(x=L_scan_F, y=cb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous'),
            showlegend=False), row=1, col=var_col)
        fig_effect_F_bif_diagram.add_trace(go.Scatter(x=L_scan_F, y=tb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous', angle=180),
            showlegend=False), row=1, col=var_col)
fig_effect_F_bif_diagram.update_xaxes(title_text='Nutrient loading L')
fig_effect_F_bif_diagram.update_yaxes(title_text='Algae A', row=1, col=1)
fig_effect_F_bif_diagram.update_yaxes(title_text='Macrophytes M', row=1, col=2)
fig_effect_F_bif_diagram.update_layout(height=400, width=800,
    legend=dict(x=0.14, y=1.15, orientation='h', entrywidth=80), template='simple_white')
fig_effect_F_bif_diagram.show()

**Bifurcation diagram in F with fixed L values**

In [ ]:
L_bif_values = [0.0, 0.9, 1.6]
L_bif_colors = [colors[3], colors[5], colors[7]]
L_bif_dashes = ['dashdot', 'dash', 'dot']
F_bif_scan = np.linspace(-0.5, 0.5, 80)
arrow_steps = [24, 26, 28]
fig_bif_F = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
for f_idx, (L_val, col_hex, dash) in enumerate(zip(L_bif_values, L_bif_colors, L_bif_dashes)):
    cb, tb = branches_by_continuation(F_bif_scan, 'F', L_val, params)
    fig_bif_F.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col_hex, dash=dash, width=2), name=f'L = {L_val}'))
    sizes = [12 if (i - f_idx) % arrow_steps[f_idx] == 0 else 0 for i in range(len(F_bif_scan))]
    for var_col, var_idx in [(1, iA), (2, iM)]:
        fig_bif_F.add_trace(go.Scatter(x=F_bif_scan, y=cb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous'),
            showlegend=False), row=1, col=var_col)
        fig_bif_F.add_trace(go.Scatter(x=F_bif_scan, y=tb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous', angle=180),
            showlegend=False), row=1, col=var_col)
fig_bif_F.update_xaxes(title_text='Fishing F')
fig_bif_F.update_yaxes(title_text='Algae A', row=1, col=1)
fig_bif_F.update_yaxes(title_text='Macrophytes M', row=1, col=2)
fig_bif_F.update_layout(height=400, width=800,
    legend=dict(x=0.13, y=1.15, orientation='h', entrywidth=80), template='simple_white')
fig_bif_F.show()

### Time series visualisation

In [ ]:
L_eq = 0.9; F_eq = 0.01
equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

dt = 0.1; sub = int(1/dt)
T_eq = 365*10  # in days
traj_clear  = simulate_sdde(equilibrium_clear,  params, T=int(T_eq/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=0.01, seed=0)[::sub]
traj_turbid = simulate_sdde(equilibrium_turbid, params, T=int(T_eq/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=0.01, seed=1)[::sub]

In [ ]:
# Plot data
def plot_trajectories_stacked(traj, state_names=state_names, title=None):
    """One subplot per variable, stacked vertically (traj columns: [L,N,A,M,Z,PF,F], or [N,A,M,Z,PF] for reduced_state_names)."""
    n = len(state_names)
    fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.02)
    t = np.arange(len(traj))
    if state_names == reduced_state_names:
        traj = traj[:, 1:]  # skip L, F and I for x-axis if not plotted
    for i, vname in enumerate(state_names):
        fig.add_trace(go.Scatter(x=t, y=traj[:, i], mode='lines',
                                 line=dict(width=0.8, color=C[vname]),
                                 name=vname, showlegend=False),
                      row=i + 1, col=1)
        ax = 'yaxis' if i == 0 else f'yaxis{i + 1}'
        side = 'right' if i % 2 else 'left'
        fig.update_layout(**{ax: dict(side=side, title=vname)})
    fig.update_xaxes(range=[0, len(traj)])
    fig.update_xaxes(title_text='Time (days)', row=n, col=1)
    fig.update_layout(height=110 * n, width=500, title_text=title, plot_bgcolor='rgba(215, 238, 244, 0.3)')
    return fig

In [ ]:
fig_clear = plot_trajectories_stacked(traj_clear, state_names=reduced_state_names, title='Trajectory from clear equilibrium')
fig_turbid = plot_trajectories_stacked(traj_turbid, state_names=reduced_state_names, title='Trajectory from turbid equilibrium')
fig_clear.show()
fig_turbid.show()

## 2. Loop-contribution decomposition

### 2.1 Analytical decomposition

In [ ]:
def jacobian(state, F, p):
    """Analytical 5x5 Jacobian of model_without_delay at `state` (order N,A,M,Z,PF)."""
    N, A, M, Z, PF = (state[iN], state[iA], state[iM], state[iZ], state[iPF]) if len(state) >= 7 \
                     else (state[0], state[1], state[2], state[3], state[4])
    fN_A = N/(p['K_A']+N); dfN_A = p['K_A']/(p['K_A']+N)**2
    fN_M = N/(p['K_M']+N); dfN_M = p['K_M']/(p['K_M']+N)**2
    sig = 1.0/(1.0+p['s']*A*A); dsig = -p['s']*2*A*sig**2
    dGA_dN = p['g_A']*dfN_A*A; dGA_dA = p['g_A']*fN_A
    dGM_dN = p['g_M']*dfN_M*M*sig; dGM_dM = p['g_M']*fN_M*sig; dGM_dA = p['g_M']*fN_M*M*dsig
    dmuA = p['m_A'] + 2*p['c_A']*A; dmuM = p['m_M'] + 2*p['c_M']*M
    # Holling II grazing derivatives
    h = p['h_A']; fr = A/(A+h); dfr = h/(A+h)**2
    dGr_dA = p['a']*dfr*Z          # d(graze)/dA
    dGr_dZ = p['a']*fr             # d(graze)/dZ
    # macrophyte refuge: pf_eff = p_F/(1+q_ref*M)
    pf = p['p_F']/(1.0+p['q_ref']*M)
    dpf_dM = -p['p_F']*p['q_ref']/(1.0+p['q_ref']*M)**2
    J = np.zeros((5,5))
    J[jN,jN] = -p['flush'] - dGA_dN - dGM_dN
    J[jN,jA] = -dGA_dA - dGM_dA + p['c_PA']*dmuA
    J[jN,jM] = -dGM_dM + p['c_PM']*dmuM
    J[jA,jN] = dGA_dN
    J[jA,jA] = dGA_dA - dGr_dA - dmuA
    J[jA,jZ] = -dGr_dZ
    J[jM,jN] = dGM_dN; J[jM,jA] = dGM_dA; J[jM,jM] = dGM_dM - dmuM
    J[jZ,jA] = p['e_Z']*dGr_dA
    J[jZ,jZ] = p['e_Z']*dGr_dZ - pf*PF - p['m_Z'] - 2*p['c_Z']*Z
    J[jZ,jPF] = -pf*Z
    J[jZ,jM] = -dpf_dM*Z*PF 
    J[jPF,jZ] = p['e_F']*pf*PF
    J[jPF,jPF] = p['e_F']*pf*Z - F - p['m_F'] - 2*p['c_F']*PF
    J[jPF,jM] = p['e_F']*dpf_dM*Z*PF
    return J

In [ ]:
def decomposition_analytical_lagged(state, F, params, T_horizon=30, lags=LAGS, dt=0.1):
    """Analytical (ground-truth) loop decomposition of the algae self-response A->A at lag T=0..T_horizon days.
    Linearises the SDDE around `state` (jacobian + per-link lags), then blocks variables to isolate
    Total, Direct, Nutrient (N,M), Trophic (Z,PF) and Mixed (remainder) contributions."""

    J   = jacobian(state, F, params)             # 5x5 continuous Jacobian (per-day rates)
    sub = int(round(1.0 / dt))                 # fine steps per day (=10 for dt=0.1)
    ts  = np.arange(T_horizon + 1)

    # Row i = destination equation, col j = source variable; order (N, A, M, Z, PF).
    Ld = lags
    LAG_DAYS = np.array([
        [Ld['l_N_N'],  Ld['l_A_N'],  Ld['l_M_N'],  0,            0            ],  # dN
        [Ld['l_N_A'],  Ld['l_A_A'],  0,            Ld['l_Z_A'],  0            ],  # dA
        [Ld['l_N_M'],  Ld['l_A_M'],  Ld['l_M_M'],  0,            0            ],  # dM
        [0,            Ld['l_A_Z'],  Ld['l_M_Z'],  Ld['l_Z_Z'],  Ld['l_PF_Z'] ],  # dZ
        [0,            0,            Ld['l_M_PF'], Ld['l_Z_PF'], Ld['l_PF_PF']],  # dPF
    ], dtype=float)

    # Day-lag -> fine-step offset, using simulate_sdde's off-by-one read correction
    # rl = max(lag_steps - 1, 0) so a d-day link lands exactly d days back after
    # daily subsampling.
    LAG_STEPS = np.maximum(np.round(LAG_DAYS / dt).astype(int) - 1, 0)
    D    = int(LAG_STEPS.max()) + 1            # history depth needed, in fine steps
    cols = np.arange(5)

    def self_response(seed, blocked):
        """Daily seed->seed self-response. Variables in `blocked` are removed from the
        dynamics (rows & cols of J zeroed => every path through them is deleted),
        seeded by a unit impulse in `seed` at day 0 with zero history before it."""
        Jb = J.copy()
        if blocked:
            b = list(blocked)
            Jb[b, :] = 0.0
            Jb[:, b] = 0.0
        buf = np.zeros((D, 5))                 # buf[k] = deviation k fine steps ago
        buf[0, seed] = 1.0                     # unit impulse at t = 0
        g = np.zeros(T_horizon + 1)
        g[0] = buf[0, seed]                    # = 1
        for n in range(1, T_horizon * sub + 1):
            read = buf[LAG_STEPS, cols[None, :]]   # read[i,j] = buf[LAG_STEPS[i,j], j]
            incr = (Jb * read).sum(axis=1)         # linearized d/dt of each variable
            new  = buf[0] + dt * incr              # explicit Euler step
            buf  = np.roll(buf, 1, axis=0)         # shift history, drop oldest
            buf[0] = new
            if n % sub == 0:                        # record once per day
                g[n // sub] = buf[0, seed]
        return g

    # A self-response and its loop partition
    Total          = self_response(jA, set())             # all paths
    Nutrient_loose = self_response(jA, {jZ, jPF})          # keep N,M routes
    Trophic_loose  = self_response(jA, {jM, jN})           # keep Z,PF routes
    Direct         = self_response(jA, {jN, jM, jZ, jPF})  # A self-decay only
    Nutrient = Nutrient_loose - Direct
    Trophic  = Trophic_loose  - Direct                         # Z,PF-only
    Mixed    = Total - Direct - Nutrient - Trophic

    return {
        'ts':         ts,
        'A_total':    Total,
        'A_direct':   Direct,
        'A_nutrient': Nutrient,
        'A_trophic':  Trophic,
        'A_mixed':    Mixed,
    }

### Validation of the linear approximation

In [ ]:
def delayed_jacobian_blocks(state, F, p, lags=LAGS):
    """Partition the Jacobian of the SDDE by lag value.

    Returns {lag: 5x5 matrix}, one block per distinct lag (0, FAST, SLOW).
    Block B[k] holds the partial derivatives of each equation with respect to
    the variables read k days in the past. Summing all blocks gives the
    conventional (instantaneous) Jacobian.

    Used by simulate_lna_delayed to set up the linearised delay system
    d(dx) = [J0 dx(t) + J1 dx(t-1) + J2 dx(t-2)] dt + noise."""
    N, A, M, Z, PF = (state[iN],state[iA],state[iM],state[iZ],state[iPF]) if len(state)==7 \
                     else (state[0],state[1],state[2],state[3],state[4])
    fN_A=N/(p['K_A']+N); dfN_A=p['K_A']/(p['K_A']+N)**2
    fN_M=N/(p['K_M']+N); dfN_M=p['K_M']/(p['K_M']+N)**2
    sig=1.0/(1.0+p['s']*A*A); dsig=-p['s']*2*A*sig**2
    dmuA=p['m_A']+2*p['c_A']*A; dmuM=p['m_M']+2*p['c_M']*M
    h=p['h_A']; fr=A/(A+h); dfr=h/(A+h)**2
    L_AA=int(lags['l_A_A']); L_MM=int(lags['l_M_M']); L_ZZ=int(lags['l_Z_Z']); L_PFPF=int(lags['l_PF_PF'])
    L0=int(lags['l_N_N']); L_AN=int(lags['l_A_N']); L_NA=int(lags['l_N_A']); L_ZA=int(lags['l_Z_A'])
    L_NM=int(lags['l_N_M']); L_AM=int(lags['l_A_M']); L_MN=int(lags['l_M_N'])
    L_AZ=int(lags['l_A_Z']); L_ZPF=int(lags['l_Z_PF']); L_PFZ=int(lags['l_PF_Z'])
    L_MZ=int(lags['l_M_Z']); L_MPF=int(lags['l_M_PF'])
    pf=p['p_F']/(1.0+p['q_ref']*M); dpf_dM=-p['p_F']*p['q_ref']/(1.0+p['q_ref']*M)**2
    B={}
    def add(lag,i,j,v):
        B.setdefault(lag,np.zeros((5,5)))[i,j]+=v
    # dN row
    add(L0,  jN,jN,-p['flush']-p['g_A']*dfN_A*A-p['g_M']*dfN_M*M*sig)
    add(L_AN,jN,jA,-p['g_A']*fN_A-p['g_M']*fN_M*M*dsig+p['c_PA']*dmuA)
    add(L_MN,jN,jM,-p['g_M']*fN_M*sig+p['c_PM']*dmuM)
    # dA row
    add(L_AA,jA,jA, p['g_A']*fN_A - p['a']*dfr*Z - dmuA)
    add(L_NA,jA,jN, p['g_A']*dfN_A*A)
    add(L_ZA,jA,jZ, -p['a']*fr)
    # dM row
    add(L_MM,jM,jM, p['g_M']*fN_M*sig - dmuM)
    add(L_NM,jM,jN, p['g_M']*dfN_M*M*sig)
    add(L_AM,jM,jA, p['g_M']*fN_M*M*dsig)
    # dZ row
    add(L_ZZ, jZ,jZ, p['e_Z']*p['a']*fr - pf*PF - p['m_Z'] - 2*p['c_Z']*Z)
    add(L_AZ, jZ,jA, p['e_Z']*p['a']*dfr*Z)
    add(L_PFZ,jZ,jPF,-pf*Z)
    add(L_MZ, jZ,jM, -dpf_dM*Z*PF)                    
    # dPF row
    add(L_PFPF,jPF,jPF, p['e_F']*pf*Z - F - p['m_F'] - 2*p['c_F']*PF)
    add(L_ZPF,jPF,jZ,  p['e_F']*pf*PF)
    add(L_MPF,jPF,jM,  p['e_F']*dpf_dM*Z*PF)    
    return B


def simulate_lna_delayed(state0, p, T, dt=0.1, L=1.0, F=0.0,
                         noise_std=0.01, seed=0):
    """Linear delay SDE around x*: dδx = [J_0 δx(t) + J_1 δx(t-1) + J_2 δx(t-2)] dt
    + diag(σ x*) dW."""
    rng   = np.random.default_rng(seed)
    sigma = np.full(5, noise_std) if np.isscalar(noise_std) else np.asarray(noise_std)
    _s = np.asarray(state0)
    xstar = np.array(_s[1:6] if len(_s) >= 7 else _s, float)
    B = delayed_jacobian_blocks(xstar, F, p)
    J0 = B.get(0, np.zeros((5,5)))
    J1 = B.get(FAST, np.zeros((5,5)))
    J2 = B.get(SLOW, np.zeros((5,5)))
    k1, k2 = int(round(FAST / dt)), int(round(SLOW / dt))
    max_lag = max(k1, k2)
    hist = np.zeros((max_lag + 1, 5))
    dx   = np.zeros(5)
    traj = np.zeros((T, 5))
    diff_amp = sigma * xstar * np.sqrt(dt)
    for n in range(T):
        traj[n] = xstar + dx
        drift = J0 @ dx + J1 @ hist[k1-1] + J2 @ hist[k2-1]   # off-by-one: read one step earlier
        eta   = rng.standard_normal(5)
        dx    = dx + dt * drift + diff_amp * eta
        hist[1:] = hist[:-1]
        hist[0]  = dx
    return traj


noise_std = 0.01
T_sim_days = 365*10 
dt_sim = 0.1
sub_daily = int(1/dt_sim)
n_lna_sims = 50
L_eq = 0.9; F_eq = 0.01
equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

def _visual_run(eq, seed_truth, seed_lna_start):
    """One nonlinear SDDE run (truth) and n_lna_sims linearised delay-SDE runs from equilibrium `eq`; daily, N,A,M,Z,PF."""
    _dyn = np.array([iN, iA, iM, iZ, iPF])
    truth = simulate_sdde(eq, params, T=int(T_sim_days/dt_sim), dt=dt_sim,
                            L_mean=L_eq, F_mean=F_eq,
                            noise_std=noise_std, seed=seed_truth)[::sub_daily][:, _dyn]
    sim_list = []
    for k in range(n_lna_sims):
        one_sim = simulate_lna_delayed(eq, params, T=int(T_sim_days/dt_sim), dt=dt_sim,
                                    L=L_eq, F=F_eq, noise_std=noise_std,
                                    seed=seed_lna_start + k)
        sim_list.append(one_sim[::sub_daily])

    sims = np.stack(sim_list)  # (n_lna_sims, T_vis_days, 5)
    return truth, sims


truth_c, sims_c = _visual_run(equilibrium_clear,  seed_truth=991, seed_lna_start=500)
truth_t, sims_t = _visual_run(equilibrium_turbid, seed_truth=992, seed_lna_start=600)

In [ ]:
t_axis = np.arange(truth_c.shape[0])
fig_vis = make_subplots(
    rows=2, cols=2,
    subplot_titles=['A: clear',  'M: clear',
                    'A: turbid', 'M: turbid'],
    shared_xaxes=True,
)
for row, (truth, sims) in enumerate([(truth_c, sims_c), (truth_t, sims_t)], start=1):
    for col, idx in enumerate([jA, jM], start=1):
        leg = (row == 1 and col == 1)
        for k in range(n_lna_sims):
            fig_vis.add_trace(
                go.Scatter(x=t_axis, y=sims[k, :, idx], mode='lines',
                           line=dict(color='rgba(218, 165, 32, 0.18)', width=0.6),
                           name='Linearized simulations',
                           showlegend=(leg and k == 0),
                           legendgroup='lna_delayed'),
                row=row, col=col)
        fig_vis.add_trace(
            go.Scatter(x=t_axis, y=truth[:, idx], mode='lines',
                       line=dict(color=C['L'], width=1.5),
                       name='nonlinear SDDE', showlegend=leg,
                       legendgroup='sdde'),
            row=row, col=col)
fig_vis.update_layout(
    height=500, width=800,
    template='plotly_white',
    legend=dict(x=0.2, y=0.6, orientation='h', entrywidth=160),)
fig_vis.update_xaxes(title_text='Time (days)', row=2, col=1)
fig_vis.update_xaxes(title_text='Time (days)', row=2, col=2)
fig_vis.show()

### 2.2 Data-driven decomposition

In [ ]:
def make_parents_dict(lags=LAGS):
    """Tigramite parents dict {variable: [(parent, -lag), ...]} built from the link lags in `lags`."""
    def edges(child, key):
        v = lags[key]
        vals = v if isinstance(v, (list, tuple, set)) else [v]
        return [(child, -int(round(l))) for l in vals]

    parents = {
        iL: [], iF: [],
        iN:  edges(iN, 'l_N_N') + edges(iM, 'l_M_N') + edges(iA, 'l_A_N') + edges(iL, 'l_L_N'),
        iA:  edges(iA, 'l_A_A') + edges(iN, 'l_N_A') + edges(iZ, 'l_Z_A'),
        iM:  edges(iM, 'l_M_M') + edges(iN, 'l_N_M') + edges(iA, 'l_A_M'),
        iZ:  edges(iZ, 'l_Z_Z') + edges(iA, 'l_A_Z') + edges(iPF, 'l_PF_Z') + edges(iM, 'l_M_Z'),
        iPF: edges(iPF, 'l_PF_PF') + edges(iZ, 'l_Z_PF') + edges(iF, 'l_F_PF') + edges(iM, 'l_M_PF'),
    }
    return parents

In [ ]:
parents = make_parents_dict()
graph = Graphs.get_graph_from_dict(parents, tau_max=5)
plot_time_series_graph(
    graph=graph, 
    var_names=state_names,
    figsize=(10, 8),
    node_size=0.05,
)
plt.show()

In [ ]:
all_parents = make_parents_dict()

In [ ]:
def causaleffect_decomposition(traj, parents=all_parents, tau_max=100):
    """Data-driven loop decomposition of the A->A response up to lag tau_max: fit a linear
    model with tigramite (LinearMediation) on N,A,M,Z,PF and split causal effects by blocked mediators.
    Returns dict with ts and A_total / A_direct / A_nutrient / A_trophic / A_mixed."""
    # Work on the 5 dynamic variables only (N,A,M,Z,PF).
    # L and F are exogenous drivers: they have no causal parents and only
    # lag-0 links (L->N, F->PF) that fall outside the lagged VAR
    # framework, so we exclude them here.
    _dyn_global = [iN, iA, iM, iZ, iPF]
    _dyn_local  = {iN: jN, iA: jA, iM: jM, iZ: jZ, iPF: jPF}
    _names5 = ['N', 'A', 'M', 'Z', 'PF']
    traj5 = traj[:, _dyn_global] if traj.shape[1] >= 7 else traj
    df = pp.DataFrame(traj5, var_names=_names5)
    # Remap parents dict to local 5-var indices, dropping L, and F parents
    parents5 = {}
    for g_idx, l_idx in _dyn_local.items():
        parents5[l_idx] = [
            (_dyn_local[p_g], lag)
            for (p_g, lag) in parents[g_idx]
            if p_g in _dyn_local  # skip iL, iF
        ]

    med = LinearMediation(dataframe=df)                    # create the mediation object on the 5-variable data
    max_lag_model = max(abs(lag) for pl in parents5.values() for (_, lag) in pl) # find the max lag in the parents dict
    med.fit_model(all_parents=parents5, tau_max=max_lag_model)   # fit at the max lag, so only max_lag samples are dropped

    phi = np.zeros((tau_max + 1, med.N, med.N))            # empty coefficient array for lags 0..tau_max
    phi[:max_lag_model + 1] = med.phi                      # copy fitted coefficients; higher lags stay zero (no such links)
    med.tau_max   = tau_max                                # extend the effect horizon without refitting
    med.phi       = phi                                    # use the zero-padded coefficients from now on
    med.psi       = med._get_psi(phi)                      # recompute total effects up to tau_max (used by get_ce)
    med.all_psi_k = med._get_all_psi_k(phi)                # recompute single-variable-blocked effects, for consistency

    ts = np.arange(tau_max + 1)
    A_total    = np.zeros(tau_max + 1)
    A_direct   = np.zeros(tau_max + 1)
    A_Nutrient = np.zeros(tau_max + 1)
    A_Trophic  = np.zeros(tau_max + 1)
    A_Mixed    = np.zeros(tau_max + 1)

   
    for tau in range(1, tau_max + 1):
        # Total causal effect
        A_total[tau] = med.get_ce(i=jA, tau=-tau, j=jA)

        # Direct effect: paths that exclude any named mediators
        A_direct[tau] = med.get_conditional_mce(i=jA, tau=-tau, j=jA,
                                                k=[jA], notk=[jM, jN, jZ, jPF])
        
        # Loop contributions
        Nut_loose = med.get_conditional_mce(i=jA, tau=-tau, j=jA, k=[jA, jM, jN], notk=[jZ, jPF])
        Tro_loose = med.get_conditional_mce(i=jA, tau=-tau, j=jA, k=[jA, jZ, jPF], notk=[jM, jN])

        A_Nutrient[tau]        = Nut_loose - A_direct[tau]
        A_Trophic[tau]         = Tro_loose - A_direct[tau]
        A_Mixed[tau]           = A_total[tau] - A_direct[tau] - A_Nutrient[tau] - A_Trophic[tau]

        result = {
            'ts':         ts,
            'A_total':    A_total,
            'A_direct':   A_direct,
            'A_nutrient': A_Nutrient,
            'A_trophic':  A_Trophic,
            'A_mixed':    A_Mixed,
        }
    return result

## 3. Loop contributions at the two equilibria

### 3.0 Helpers

In [ ]:
def detrend_traj(traj, todetrend=[iN, iA, iM, iZ, iPF], method='linear', window=100):
    """Remove a linear trend or a centred moving average from the listed columns of traj."""
    out = traj.copy()
    dyn_cols = todetrend  # only detrend the specified columns; L and F are constant exogenous inputs
    for i in dyn_cols:
        if method == 'linear':
            # fit a degree-1 polynomial to col i and subtract it, removing the linear trend
            out[:, i] = traj[:, i] - np.polyval(np.polyfit(np.arange(len(traj)), traj[:, i], 1), np.arange(len(traj)))
        elif method == 'moving_average':
            out[:, i] = traj[:, i] - pd.Series(traj[:, i]).rolling(window=window, center=True, min_periods=1).mean().values
    return out

In [ ]:
LOOPS = [
    ("A_total",    "Total"),
    ("A_direct",   "Direct"),
    ("A_nutrient", "Nutrient"),
    ("A_trophic",  "Trophic"),
    ("A_mixed",    "Mixed"),
]
LOOP_KEYS = [k for k, _ in LOOPS]
LOOP_LABEL = dict(LOOPS)

In [ ]:
def stays_in_state(traj, low=None, high=None, window=100):
    """True if the yearly-mean A stays below `high` and/or above `low` in every block.
 
    Checking block means (not just the overall mean) catches runs that
    switch state partway through, which an overall mean can hide.
    Set window=len(traj) to fall back to a single overall mean.
    """
    A = traj[:, iA]
    n_blocks = max(len(A) // window, 1)
    blocks = A[: n_blocks * window].reshape(n_blocks, -1).mean(axis=1)
    ok = np.ones(n_blocks, dtype=bool)
    if high is not None:
        ok &= blocks < high
    if low is not None:
        ok &= blocks > low
    return bool(ok.all())
 
 
default_state_checks = {
    "clear":  lambda traj: stays_in_state(traj, high=1.0),
    "turbid": lambda traj: stays_in_state(traj, low=2.0),
}

In [ ]:
def simulate_and_decompose(sim_init, seed, params, T_max, *,
                           L_eq, F_eq, dt=0.1,
                           burnin_days=365, T_long_days=365 * 10,
                           noise_std=0.01, lags=None, keep_trajectory=True,
                           sim_fn=simulate_sdde, sim_kwargs=None,
                           assumed_parents=all_parents,
                           state_check=None):
    """One noisy realization -> detrend -> data-driven decomposition."""
    if lags is None:
        lags = LAGS
    if sim_kwargs is None:
        sim_kwargs = {}
    sub = int(round(1.0 / dt))

    traj = sim_fn(
        sim_init, params, int((burnin_days + T_long_days) / dt),
        dt=dt, L_mean=L_eq, F_mean=F_eq,
        noise_std=noise_std, sigma_L=0, sigma_F=0,
        seed=seed, lags=lags, **sim_kwargs,
    )[::sub][burnin_days:]

    if state_check is not None and not state_check(traj):
        return None

    det = detrend_traj(traj)
    dd = causaleffect_decomposition(det, tau_max=T_max, parents=assumed_parents)

    out = {"decomposition": dd}
    if keep_trajectory:
        out["traj"] = traj
        out["traj_detrended"] = det
    return out

In [ ]:
def run_decompositions(conditions, params, T_max, *,
                       n_seeds=50, seed_start=0, max_attempts=1000,
                       state_checks=default_state_checks,
                       L_eq=0.9, F_eq=0.01, dt=0.1,
                       burnin_days=365, T_long_days=365 * 10,
                       noise_std=0.01, lags=LAGS,
                       sim_fn=simulate_sdde, sim_kwargs=None,
                       assumed_parents=all_parents,
                       verbose=False):
    """Analytical decomposition once per condition; data-driven for
    `n_seeds` accepted realizations per condition.
 
    A realization is accepted only if state_checks[cond](traj) is True.
    """
 
    analytical, datadriven, trajectories = {}, {}, {}
    accepted_seeds, rejected_seeds = {}, {}
 
    for i, (cond, equilibrium_init) in enumerate(conditions.items()):
        ana = decomposition_analytical_lagged(
            equilibrium_init, F_eq, params,
            T_horizon=T_max, lags=lags, dt=dt,
        )
        analytical[cond] = ana
        ts_ref = np.asarray(ana["ts"])
 
        check = state_checks.get(cond)
 
        datadriven[cond], trajectories[cond] = {}, {}
        accepted_seeds[cond], rejected_seeds[cond] = [], []
 
        for k in range(max_attempts):
            if len(accepted_seeds[cond]) >= n_seeds:
                break
            seed = seed_start + k
            res = simulate_and_decompose(
                equilibrium_init, seed, params, T_max,
                L_eq=L_eq, F_eq=F_eq, dt=dt,
                burnin_days=burnin_days, T_long_days=T_long_days,
                noise_std=noise_std, lags=lags,
                sim_fn=sim_fn, sim_kwargs=sim_kwargs,
                assumed_parents=assumed_parents,
                state_check=check,
            )
            if res is None:
                rejected_seeds[cond].append(seed)
                if verbose:
                    print(f"  {cond:7s} seed {seed} REJECTED (left state)")
                continue
 
            accepted_seeds[cond].append(seed)
            datadriven[cond][seed] = res["decomposition"]
            trajectories[cond][seed] = {
                "traj": res["traj"],
                "traj_detrended": res["traj_detrended"]}
 
        n_acc, n_rej = len(accepted_seeds[cond]), len(rejected_seeds[cond])
        if n_acc < n_seeds:
            warnings.warn(f"'{cond}': only {n_acc}/{n_seeds} accepted "
                          f"after {max_attempts} attempts.")
        if verbose or n_rej:
            print(f"{cond}: {n_acc} accepted, {n_rej} rejected "
                  f"(rejection rate {n_rej / max(n_acc + n_rej, 1):.1%})")
 
    store = dict(ts=ts_ref, conditions=list(conditions.keys()),
                 trajectories=trajectories,
                 seeds=accepted_seeds, 
                 rejected_seeds=rejected_seeds,
                 noise_std=noise_std,
                 analytical=analytical, datadriven=datadriven,
                 sim_fn=getattr(sim_fn, "__name__", str(sim_fn)),
                 sim_kwargs=dict(sim_kwargs or {}))

    return store

In [ ]:
def plot_all_trajectories_stacked(store, state_names=reduced_state_names, conditions=None,
                                  detrended=False, show_mean=True):
    """Overlay every accepted seed's trajectory from `run_decompositions`.
 
    One row per variable, one column per condition. Individual seeds are
    thin, semi-transparent lines; the ensemble mean (optional) is drawn
    on top in black.
    """
 
    conditions = list(conditions or store["conditions"])
    key = "traj_detrended" if detrended else "traj"
    n, n_cols = len(state_names), len(conditions)
    # Same column logic as plot_trajectories_stacked
    col_offset = 1 if state_names == reduced_state_names else 0
 
    fig = make_subplots(rows=n, cols=n_cols, shared_xaxes=True,
                        vertical_spacing=0.04, horizontal_spacing=0.08,
                        column_titles=conditions)
 
    for j, cond in enumerate(conditions):
        runs = store["trajectories"][cond]
        seeds = list(runs.keys())
        if not seeds:
            continue
        # (n_seeds, T, n_vars), trimmed to the shortest run just in case
        T = min(len(runs[s][key]) for s in seeds)
        stack = np.stack([np.asarray(runs[s][key])[:T, col_offset:] for s in seeds])
        t = np.arange(T)
 
        for i, vname in enumerate(state_names):
            for s, tr in zip(seeds, stack):
                fig.add_trace(go.Scattergl(
                    x=t, y=tr[:, i], mode="lines",
                    line=dict(width=0.3, color=C[vname]),
                    opacity=0.3, name=f"{cond} seed {s}",
                    hoverinfo="skip", showlegend=False),
                    row=i + 1, col=j + 1)
            if show_mean:
                fig.add_trace(go.Scattergl(
                    x=t, y=stack[:, :, i].mean(axis=0), mode="lines",
                    line=dict(width=1.2, color="black"),
                    name=f"{cond} mean", showlegend=False),
                    row=i + 1, col=j + 1)
            if j == 0:
                fig.update_yaxes(title_text=vname, row=i + 1, col=1)
 
        fig.update_xaxes(range=[0, T], col=j + 1)
        fig.update_xaxes(title_text="Time (days)", row=n, col=j + 1)
 
    fig.update_layout(height=110 * n, width=500 * n_cols,
                      plot_bgcolor="rgba(215, 238, 244, 0.3)")
    return fig
 

In [ ]:
# plots
def _to_rgba(color, alpha):
    """Accept 'rgb(r,g,b)' or '#rrggbb' and return 'rgba(r,g,b,alpha)'."""
    if color.startswith("rgb("):
        r, g, b = color[4:-1].split(",")
        return f"rgba({int(r)},{int(g)},{int(b)},{alpha})"
    h = color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def plot_loop_timeseries(store, *, conditions=None,
                         left_loops=("A_total", "A_direct"),
                         right_loops=("A_nutrient", "A_trophic", "A_mixed"),
                         realisation_alpha=0.20,
                         column_widths=(0.40, 0.60),
                         horizontal_spacing=0.10, vertical_spacing=0.12,
                         show_seed_mean=True, show_realisation_legend=False,
                         exclude_seeds=None,
                         height=720, width=1100):
    """Single figure, one row per condition, two panels per row (2x2):
      left  = Total & Direct;  right = Nutrient, Trophic, Mixed.
    Each loop in its C[...] color: faint realisations, dotted mean, solid analytical.
    Legend shown once (top row)."""
    ts = store["ts"]
    mask = ts >= 1          # drop the T=0 sample (use ts > 0 if you prefer)
    ts_plot = ts[mask]
    if conditions is None:
        conditions = store["conditions"]
    conditions = list(conditions)
    nrows = len(conditions)

    titles = []
    for cond in conditions:
        titles += [f"{cond}", f"{cond}"]

    fig = make_subplots(rows=nrows, cols=2, subplot_titles=titles,
                        shared_xaxes=True, column_widths=list(column_widths),
                        horizontal_spacing=horizontal_spacing,
                        vertical_spacing=vertical_spacing)

    for r, cond in enumerate(conditions, start=1):
        first_row = (r == 1)
        seeds = store["seeds"][cond] if isinstance(store["seeds"], dict) else store["seeds"]
        drop = (exclude_seeds or {}).get(cond, [])                 # seeds to exclude for this condition
        seeds = [s for s in seeds if s not in drop]                # removed from realisations AND mean
        for col, loops in ((1, left_loops), (2, right_loops)):
            for k in loops:
                color = C[k]
                faint = _to_rgba(color, realisation_alpha)
                label = LOOP_LABEL[k]
                curves = np.stack([store["datadriven"][cond][s][k]
                                   for s in seeds], axis=0)

                # realisations
                for i in range(curves.shape[0]):
                    fig.add_trace(go.Scatter(
                        x=ts_plot, y=curves[i][mask], mode="lines",
                        line=dict(color=faint, width=1),
                        name=f"{label} (realisations)", legendgroup=f"{k}-real",
                        showlegend=(show_realisation_legend and first_row and i == 0),
                        hoverinfo="skip",
                    ), row=r, col=col)

                # analytical
                fig.add_trace(go.Scatter(
                    x=ts_plot, y=store["analytical"][cond][k][mask], mode="lines",
                    line=dict(color=color, width=2.6),
                    name=f"{label} (analytical)", legendgroup=f"{k}-ana",
                    showlegend=first_row,
                ), row=r, col=col)

                # mean
                if show_seed_mean:
                    fig.add_trace(go.Scatter(
                        x=ts_plot, y=curves.mean(axis=0)[mask], mode='lines+markers',
                        marker=dict(color=color, size=5, symbol='circle-open'),
                        line=dict(color=color, width=2),
                        name=f"{label} (from data)", legendgroup=f"{k}-mean",
                        showlegend=first_row,
                    ), row=r, col=col)

    for r in range(1, nrows + 1):
        for c in (1, 2):
            fig.add_hline(y=0, line=dict(color="gray", width=0.5), row=r, col=c)

    fig.update_xaxes(title_text="T (days)", range=[0, float(ts.max())], row=nrows)
    fig.update_yaxes(title_text="Loop strength at lag T", col=1)
    fig.update_layout(
        template="plotly_white", height=height, width=width,
        legend=dict(font=dict(size=10)),
    )
    return fig

In [ ]:
# error measures (computed from stored decompositions)

def rmse(a, d):
    """Root-mean-square distance between analytical `a` and data-driven `d`
    (loop-contribution units: the typical vertical gap between the two curves).
    Error is computed from t=1: index 0 is a spurious mismatch (analytical
    self-response has g[0]=1 while causaleffect only fills tau>=1)."""
    a = np.asarray(a, float)[1:]; d = np.asarray(d, float)[1:]   # error is computed from t=1
    return np.sqrt(np.mean((d - a) ** 2))


def _summarize(vals):
    """Median, quartiles, IQR and count of vals (NaNs ignored)."""
    vals = np.asarray(vals, float)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return dict(median=np.nan, q1=np.nan, q3=np.nan, iqr=np.nan, n=0)
    q1, med, q3 = np.percentile(vals, [25, 50, 75])
    return dict(median=med, q1=q1, q3=q3, iqr=q3 - q1, n=vals.size)


def compute_errors(store, *, loop_keys=LOOP_KEYS, exclude_seeds=None):
    """Per-seed RMSE of each data-driven loop curve vs the analytical curve
    (over all lags, from t=1), plus median/IQR across seeds.

    Returns:
        { 'seeds':   { cond: [seeds used] },
          'errors':  { cond: { loop: np.array([err per seed]) } },
          'summary': { cond: { loop: dict(median,q1,q3,iqr,n) } } }
    """
    errors, summary, used_seeds = {}, {}, {}
    for cond in store["conditions"]:
        ana = store["analytical"][cond]
        errors[cond], summary[cond] = {}, {}
        seeds = store["seeds"][cond] if isinstance(store["seeds"], dict) else store["seeds"]
        drop = (exclude_seeds or {}).get(cond, [])                 # seeds to exclude for this condition
        seeds = [s for s in seeds if s not in drop]                # removed from errors, median and IQR
        used_seeds[cond] = seeds
        for k in loop_keys:
            e = np.array([rmse(ana[k], store["datadriven"][cond][s][k]) for s in seeds], float)
            errors[cond][k] = e
            summary[cond][k] = _summarize(e)

    return dict(seeds=used_seeds, errors=errors, summary=summary)

### 3.1 Base case: Figure 4

In [ ]:
L_eq = 0.9
F_eq = 0.01
equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

In [ ]:
dt = 0.1; sub = int(1/dt); noise_std=0.01
burnin = 365*1; T_long = 365*10  # in days
traj_c_long = simulate_sdde(equilibrium_clear,  params, int((burnin+T_long)/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=noise_std, sigma_L=0, sigma_F=0, seed=3)[::sub][burnin:]
traj_t_long = simulate_sdde(equilibrium_turbid, params, int((burnin+T_long)/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=noise_std, sigma_L=0, sigma_F=0, seed=1)[::sub][burnin:]

In [ ]:
figC = plot_trajectories_stacked(traj_c_long, reduced_state_names, title='Clear')
figT = plot_trajectories_stacked(traj_t_long, reduced_state_names, title='Turbid')
# figC.show()
# figT.show()

With n_seeds=50, the following cell takes approximately 10min to compute.

In [ ]:
# conditions = {
#     "clear":  equilibrium_clear,
#     "turbid": equilibrium_turbid,
# }

# dt = 0.1; sub = int(1/dt); noise_std=0.01
# T_max = 100
# n_seeds = 50
# T_long_days = 365 * 10
# burnin_days = 365
# parents = make_parents_dict(LAGS)
# store_base = run_decompositions(conditions, params, T_max=T_max, T_long_days=T_long_days, burnin_days=burnin_days, n_seeds=n_seeds, assumed_parents=parents)
# with open("store_base.pkl", "wb") as f:
#     pickle.dump(store_base, f)

In [ ]:
with open("store_base.pkl", "rb") as f:
    store_base = pickle.load(f)

In [ ]:
fig = plot_all_trajectories_stacked(store_base, detrended=False) 
fig.show()

In [ ]:
fig_base = plot_loop_timeseries(store_base)
fig_base.show()
# fig_base.write_image('figures/fig_base.svg')

In [ ]:
error_base = compute_errors(store_base)

### 3.2 Error evolution with time

In [ ]:
def plot_error_vs_time(store,
                       left_loops=("A_total", "A_direct"),
                       right_loops=("A_nutrient", "A_trophic", "A_mixed")):
    """Per-point error vs time horizon, figure-4 layout (one row per condition,
    Total&Direct left, other loops right). One solid line per loop:

        RMSE(t) = sqrt( mean_s [ d_s(t) - a(t) ]^2 )   (across-seed RMS at t).
    """
    ts = np.asarray(store["ts"], float)
    conditions = list(store["conditions"])
    nrows = len(conditions)

    titles = []
    for cond in conditions:
        titles += [f"{cond}", f"{cond}"]

    fig = make_subplots(rows=nrows, cols=2, subplot_titles=titles,
                        shared_xaxes=True, column_widths=(0.40, 0.60),
                        horizontal_spacing=0.10,
                        vertical_spacing=0.12)

    mask = ts >= 1  # start computing the error at t=1

    for r, cond in enumerate(conditions, start=1):
        first_row = (r == 1)
        for col, loops in ((1, left_loops), (2, right_loops)):
            for k in loops:
                color = C.get(k)
                label = LOOP_LABEL[k]
                a = np.asarray(store["analytical"][cond][k], float)
                D = np.stack([store["datadriven"][cond][s][k]
                              for s in store["seeds"][cond]], axis=0)
                rmse_t = np.sqrt(np.mean((D - a[None, :]) ** 2, axis=0))

                fig.add_trace(go.Scatter(
                    x=ts[mask], y=rmse_t[mask], mode="lines",
                    line=dict(color=color, width=2.4),
                    name=label, legendgroup=k, showlegend=first_row),
                    row=r, col=col)

    fig.update_xaxes(title_text="T (days)", range=[0, float(ts.max())], row=nrows)
    fig.update_yaxes(title_text="RMSE across seeds", col=1)
    fig.update_layout(template="plotly_white", height=720, width=1100)
    return fig

In [ ]:
fig_error_vs_time = plot_error_vs_time(store_base)
fig_error_vs_time.show()
# fig_error_vs_time.write_image('figures/fig_error_vs_time.svg')

### 3.3 Smaller sample size (=1000, =100)

In [ ]:
L_eq = 0.9
F_eq = 0.01
equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

In [ ]:
conditions = {
    "clear":  equilibrium_clear,
    "turbid": equilibrium_turbid,
}

**1000 data points**

In [ ]:
# T_max = 100
# T_long_days = 1000
# burnin_days = 100
# n_seeds = 50
# store_1000samples = run_decompositions(conditions, params, T_max=T_max, T_long_days=T_long_days, burnin_days=burnin_days, n_seeds=n_seeds)

# with open("store_1000samples.pkl", "wb") as f:
#     pickle.dump(store_1000samples, f)

In [ ]:
with open("store_1000samples.pkl", "rb") as f:
    store_1000samples = pickle.load(f)

In [ ]:
fig = plot_all_trajectories_stacked(store_1000samples, detrended=False) 
fig.show()

In [ ]:
fig_1000samples = plot_loop_timeseries(store_1000samples)
fig_1000samples.show()
# fig_1000samples.write_image('figures/fig_1000samples.svg')

In [ ]:
error_1000samples = compute_errors(store_1000samples)

**100 data points**

In [ ]:
# dt = 0.1; sub = int(1/dt); noise_std=0.01
# T_max = 100
# T_long_days = 100
# burnin_days = 50
# n_seeds = 50
# store_100samples = run_decompositions(conditions, params, T_max=T_max, T_long_days=T_long_days, burnin_days=burnin_days, n_seeds=n_seeds)

# with open("store_100samples.pkl", "wb") as f:
#     pickle.dump(store_100samples, f)

In [ ]:
with open("store_100samples.pkl", "rb") as f:
    store_100samples = pickle.load(f)

In [ ]:
fig = plot_all_trajectories_stacked(store_100samples, detrended=False) 
fig.show()

In [ ]:
fig_100samples = plot_loop_timeseries(store_100samples)
fig_100samples.show()
# fig_100samples.write_image('figures/fig_100samples.svg')

In [ ]:
cond = "turbid"
exploding = [s for s in store_100samples["seeds"][cond]
       if store_100samples["datadriven"][cond][s]["A_trophic"][-1] > 5]
print(exploding)

In [ ]:
fig_100samples_ExplodingExcluded = plot_loop_timeseries(store_100samples, exclude_seeds={cond: exploding})
fig_100samples_ExplodingExcluded.show()
# fig_100samples_ExplodingExcluded.write_image('figures/fig_100samples_ExplodingExcluded.svg')

In [ ]:
error_100samples = compute_errors(store_100samples)
error_100samples_ExplodingExcluded = compute_errors(store_100samples, exclude_seeds={cond: exploding})

### 3.4 High noise

In [ ]:
L_eq = 0.9
F_eq = 0.01

equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

In [ ]:
# conditions = {
#     "clear":  equilibrium_clear,
#     "turbid": equilibrium_turbid,
# }

# T_max = 100
# T_long_days = T_long
# burnin_days = burnin
# noise_std = 0.05
# n_seeds = 50
# store_highNoise = run_decompositions(conditions, params, T_max=T_max, T_long_days=T_long_days, burnin_days=burnin_days, 
#                                      noise_std=noise_std, n_seeds=n_seeds, verbose=True)
# with open("store_highNoise.pkl", "wb") as f:
#     pickle.dump(store_highNoise, f)

In [ ]:
with open("store_highNoise.pkl", "rb") as f:
    store_highNoise = pickle.load(f)

In [ ]:
fig = plot_all_trajectories_stacked(store_highNoise, detrended=False)
fig.show()

In [ ]:
fig_highNoise = plot_loop_timeseries(store_highNoise)
fig_highNoise.show()
# fig_highNoise.write_image('figures/fig_highNoise.svg')

In [ ]:
error_highNoise = compute_errors(store_highNoise)

### 3.5 Unobserved confounder (as a function of the correlation coefficient between noise terms)

In [ ]:
_confounded_idx= {'N': 0, 'A': 1, 'M': 2, 'Z': 3, 'PF': 4}  # indices into the 5-vector [N,A,M,Z,PF]


def simulate_sdde_confounding(state0, p, T, dt=0.1, L_mean=1.0, F_mean=0.0,
                  noise_std=0.01, sigma_L=0, sigma_F=0,
                  seed=0, lags=LAGS, additive_noise=False,
                  confounded=None, rho=0.5, conf_phi=0.7, return_latent=False):
    """Same SDDE as simulate_sdde, but the noise of the variables in `confounded` (e.g. ['A','M'])
    shares a hidden AR(1) common cause U: pairwise noise correlation rho, persistence conf_phi.
    Returns traj (T, 7) and the latent series if return_latent."""
    
    rng = np.random.default_rng(seed)
    s0 = np.asarray(state0, float); s0_5 = s0[1:6] if len(s0) >= 7 else s0
    L0 = float(np.mean(L_mean)); F0 = float(np.mean(F_mean))
    lags_dt = {k: int(round(v/dt)) for k, v in lags.items()}
    rl = {k: max(v-1, 0) for k, v in lags_dt.items()}
    max_lag = max(lags_dt.values())
    hist = np.tile(np.array([L0, *s0_5, F0], float), (max_lag+1, 1))
    traj = np.zeros((T, 7)); state = s0_5.copy(); sigma = np.full(5, noise_std)
    L_n_arr = (np.full(T, L_mean, float) if np.isscalar(L_mean) else np.asarray(L_mean, float)) + sigma_L*rng.standard_normal(T)
    F_n_arr = (np.full(T, F_mean, float) if np.isscalar(F_mean) else np.asarray(F_mean, float)) + sigma_F*rng.standard_normal(T)

    # --- latent confounder setup ------------------------------------------
    # One hidden common cause U loads on the named endogenous vars. Each
    # member's standardized shock: eta_i = sqrt(rho)*U + sqrt(1-rho)*e_i, so
    # every pair has corr = rho and marginal variance stays 1 (correlation
    # only, not noise magnitude). U is an OU/AR(1) latent with unit stationary
    # variance: U_t = phi*U_{t-1} + sqrt(1-phi^2)*xi_t, so the shared shock
    # persists to the next step and contaminates lag-1 cross structure.
    use_conf = bool(confounded) and rho > 0
    if bool(confounded):
        idx = [_confounded_idx[v] for v in confounded]     # index into [N,A,M,Z,PF]
        if len(idx) < 2:
            raise ValueError("confounded needs >= 2 endogenous variables to induce confounding")
        if not 0.0 <= rho <= 1.0:
            raise ValueError("rho must be in [0, 1]")
        if not abs(conf_phi) < 1.0:
            raise ValueError("conf_phi must satisfy |phi| < 1 for a stationary latent")
        lam = np.zeros(5); lam[idx] = np.sqrt(rho)
        idio = np.sqrt(1.0 - lam**2)
        phi2 = conf_phi*conf_phi
        U = rng.standard_normal()                    # start at stationary variance
    U_series = np.zeros(T) if return_latent else None
    # ----------------------------------------------------------------------

    for n in range(T):
        N, A, M, Z, PF = state
        L_n, F_n = L_n_arr[n], F_n_arr[n]
        traj[n] = [L_n, N, A, M, Z, PF, F_n]
        N_for_A = hist[rl['l_N_A'], iN]
        N_for_M = hist[rl['l_N_M'], iN]
        A_for_M = hist[rl['l_A_M'], iA]
        Z_for_A = hist[rl['l_Z_A'], iZ]
        A_for_Z = hist[rl['l_A_Z'], iA]
        M_for_N = hist[rl['l_M_N'], iM]
        PF_for_Z = hist[rl['l_PF_Z'], iPF]
        Z_for_PF = hist[rl['l_Z_PF'], iZ]
        M_for_Z = hist[rl['l_M_Z'], iM]
        M_for_PF = hist[rl['l_M_PF'], iM]
        N_for_N = hist[rl['l_N_N'], iN]
        A_for_N = hist[rl['l_A_N'], iA]
        A_self  = hist[rl['l_A_A'],   iA]
        M_self  = hist[rl['l_M_M'],   iM]
        Z_self  = hist[rl['l_Z_Z'],   iZ]
        PF_self = hist[rl['l_PF_PF'], iPF]
        # dN
        GA_s = p['g_A']*N_for_N/(p['K_A']+N_for_N)*A_for_N
        GM_s = p['g_M']*N_for_N/(p['K_M']+N_for_N)*M_for_N/(1.0+p['s']*A_for_N**2)
        muA_N = p['m_A']*A_for_N + p['c_A']*A_for_N**2
        muM_N = p['m_M']*M_for_N + p['c_M']*M_for_N**2
        dN = L_n - p['flush']*N_for_N - GA_s - GM_s + p['c_PA']*muA_N + p['c_PM']*muM_N + p['eps_N']
        # dA
        GA = p['g_A']*N_for_A/(p['K_A']+N_for_A)*A_self
        graze = p['a']*A_self/(A_self+p['h_A'])*Z_for_A
        muA = p['m_A']*A_self + p['c_A']*A_self**2
        dA = p['eps_A'] + GA - graze - muA
        # dM
        GM = p['g_M']*N_for_M/(p['K_M']+N_for_M)*M_self/(1.0+p['s']*A_for_M**2)
        muM = p['m_M']*M_self + p['c_M']*M_self**2
        dM = p['eps_M'] + GM - muM
        # dZ
        gain = p['e_Z']*p['a']*A_for_Z/(A_for_Z+p['h_A'])*Z_self
        pf_eff_Z = p['p_F']/(1.0+p['q_ref']*M_for_Z)
        dZ = p['eps_Z'] + gain - pf_eff_Z*Z_self*PF_for_Z - p['m_Z']*Z_self - p['c_Z']*Z_self**2
        # dPF
        pf_eff_PF = p['p_F']/(1.0+p['q_ref']*M_for_PF)
        dPF = p['eps_PF'] + p['e_F']*pf_eff_PF*Z_for_PF*PF_self - F_n*PF_self - p['m_F']*PF_self - p['c_F']*PF_self**2
        drift = np.array([dN, dA, dM, dZ, dPF])

        # --- correlated (confounded) vs independent shocks ---
        if use_conf:
            e = rng.standard_normal(5)
            eta = lam*U + idio*e
            if return_latent:
                U_series[n] = U
            U = conf_phi*U + np.sqrt(1.0 - phi2)*rng.standard_normal()   # advance OU latent
        else:
            eta = rng.standard_normal(5)
        # -----------------------------------------------------

        incr = sigma*np.sqrt(dt)*eta if additive_noise else sigma*state*np.sqrt(dt)*eta
        new = state + dt*drift + incr
        hist[1:] = hist[:-1]; hist[0] = [L_n, *new, F_n]; state = new

    return (traj, U_series) if return_latent else traj

In [ ]:
L_eq = 0.9
F_eq = 0.01

equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

**Weak confounding**

In [ ]:
# conditions = {
#     "clear":  equilibrium_clear,
#     "turbid": equilibrium_turbid,
# }

# T_max = 100
# dt = 0.1; sub = int(1/dt); noise_std=0.01
# burnin_days = 365*1; T_long_days = 365*10  # in days
# rho = 0.25
# conf_phi = 0.7
# confounded = ['A', 'M', 'Z']
# n_seeds = 50

# store_confounding_rho025_phi07 = run_decompositions(
#     conditions, params, T_max,T_long_days=T_long_days, burnin_days=burnin_days, 
#     noise_std=noise_std, n_seeds=n_seeds,
#     sim_fn=simulate_sdde_confounding,
#     sim_kwargs=dict(confounded=confounded, rho=rho, conf_phi=conf_phi),
# )

# with open("store_confounding_rho025_phi07.pkl", "wb") as f:
#     pickle.dump(store_confounding_rho025_phi07, f)

In [ ]:
with open("store_confounding_rho025_phi07.pkl", "rb") as f:
    store_confounding_rho025_phi07 = pickle.load(f)

In [ ]:
fig = plot_all_trajectories_stacked(store_confounding_rho025_phi07, detrended=False)
fig.show()

In [ ]:
fig_confounding_rho025_phi07 = plot_loop_timeseries(store_confounding_rho025_phi07)
fig_confounding_rho025_phi07.show()
# fig_confounding_rho025_phi07.write_image('figures/fig_confounding_rho025_phi07.svg')

In [ ]:
error_confounding_rho025_phi07 = compute_errors(store_confounding_rho025_phi07)

**Strong confounding**

In [ ]:
# conditions = {
#     "clear":  equilibrium_clear,
#     "turbid": equilibrium_turbid,
# }

# T_max = 100
# dt = 0.1; sub = int(1/dt); noise_std=0.01
# burnin_days = 365*1; T_long_days = 365*10  # in days
# rho = 0.8
# conf_phi = 0.7
# confounded = ['A', 'M', 'Z']
# n_seeds = 50

# store_confounding_rho08_phi07 = run_decompositions(
#     conditions, params, T_max,T_long_days=T_long_days, burnin_days=burnin_days, 
#     noise_std=noise_std, n_seeds=n_seeds,
#     sim_fn=simulate_sdde_confounding,
#     sim_kwargs=dict(confounded=confounded, rho=rho, conf_phi=conf_phi),
#     verbose=False
# )

# with open("store_confounding_rho08_phi07.pkl", "wb") as f:
#     pickle.dump(store_confounding_rho08_phi07, f)

In [ ]:
with open("store_confounding_rho08_phi07.pkl", "rb") as f:
    store_confounding_rho08_phi07 = pickle.load(f)

In [ ]:
fig = plot_all_trajectories_stacked(store_confounding_rho08_phi07, detrended=False)
fig.show()

In [ ]:
fig_confounding_rho08_phi07 = plot_loop_timeseries(store_confounding_rho08_phi07)
fig_confounding_rho08_phi07.show()
# fig_confounding_rho08_phi07.write_image('figures/fig_confounding_rho08_phi07.svg')

In [ ]:
error_confounding_rho08_phi07 = compute_errors(store_confounding_rho08_phi07)

### 3.6 Lag misspecification / lag uncertainty

In [ ]:
L_eq = 0.9
F_eq = 0.01

equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

**All lags are set to 3, self-lags to 1**

In [ ]:
# conditions = {
#     "clear":  equilibrium_clear,
#     "turbid": equilibrium_turbid,
# }

# T_max = 100
# dt = 0.1; sub = int(1/dt); noise_std=0.01
# burnin_days = 365*1; T_long_days = 1000  # in days
# n_seeds = 50

# NEW = 3
# LAGS_WRONG = {
#     'l_L_N':   0,
#     'l_N_N':   1, 'l_M_N': NEW, 'l_A_N': NEW,
#     'l_N_A':   NEW, 'l_A_A': 1, 'l_Z_A': NEW,
#     'l_N_M':   NEW, 'l_A_M': NEW, 'l_M_M': 1, 
#     'l_A_Z':   NEW, 'l_Z_Z': 1, 'l_PF_Z': NEW, 'l_M_Z': NEW,
#     'l_Z_PF':  NEW, 'l_PF_PF': 1, 'l_M_PF': NEW,
#     'l_F_PF':  0,
# }

# assumed_parents = make_parents_dict(LAGS_WRONG)

# store_lags_all3_self1 = run_decompositions(
#     conditions, params, T_max,T_long_days=T_long_days, burnin_days=burnin_days, 
#     noise_std=noise_std, n_seeds=n_seeds,
#     sim_fn=simulate_sdde,
#     assumed_parents = assumed_parents,
# )

# with open("store_lags_all3_self1.pkl", "wb") as f:
#     pickle.dump(store_lags_all3_self1, f)

In [ ]:
with open("store_lags_all3_self1.pkl", "rb") as f:
    store_lags_all3_self1 = pickle.load(f)

In [ ]:
fig_lags_allFast = plot_loop_timeseries(store_lags_all3_self1)
fig_lags_allFast.show()
# fig_lags_allFast.write_image('figures/fig_lags_all3_self1.svg')

In [ ]:
error_lags_all3_self1 = compute_errors(store_lags_all3_self1)

**All lags are set to {1,2,3}**

In [ ]:
# conditions = {
#     "clear":  equilibrium_clear,
#     "turbid": equilibrium_turbid,
# }

# T_max = 100
# dt = 0.1; sub = int(1/dt); noise_std=0.01
# burnin_days = 365*1; T_long_days = 365*10  # in days
# n_seeds = 50

# LAG_SET = [1, 2, 3]
# LAGS_UNCERTAINTY = {
#     'l_L_N':   0,                                           # exogenous
#     'l_N_N':   LAG_SET, 'l_M_N': LAG_SET, 'l_A_N': LAG_SET, # -> N
#     'l_N_A':   LAG_SET, 'l_A_A': LAG_SET, 'l_Z_A': LAG_SET, # -> A
#     'l_N_M':   LAG_SET, 'l_A_M': LAG_SET, 'l_M_M': LAG_SET, # -> M
#     'l_A_Z':   LAG_SET, 'l_Z_Z': LAG_SET, 'l_PF_Z': LAG_SET, 'l_M_Z': LAG_SET,  # -> Z
#     'l_Z_PF':  LAG_SET, 'l_PF_PF': LAG_SET, 'l_M_PF': LAG_SET,                   # -> PF
#     'l_F_PF':  0,                                           # exogenous
# }

# assumed_parents = make_parents_dict(LAGS_UNCERTAINTY)

# store_lags_uncertainty = run_decompositions(
#     conditions, params, T_max,T_long_days=T_long_days, burnin_days=burnin_days, 
#     noise_std=noise_std, n_seeds=n_seeds,
#     sim_fn=simulate_sdde,
#     assumed_parents = assumed_parents,
# )

# with open("store_lags_uncertainty.pkl", "wb") as f:
#     pickle.dump(store_lags_uncertainty, f)

In [ ]:
with open("store_lags_uncertainty.pkl", "rb") as f:
    store_lags_uncertainty = pickle.load(f)

In [ ]:
fig_lags_uncertainty = plot_loop_timeseries(store_lags_uncertainty)
fig_lags_uncertainty.show()
# fig_lags_uncertainty.write_image('figures/fig_lags_uncertainty.svg')

In [ ]:
error_lags_uncertainty = compute_errors(store_lags_uncertainty)

### 3.7 Error comparison across experiments

In [ ]:
def plot_error_comparison(error_results, conditions=("clear", "turbid")):
    """Compare several error results (variations) against a base, per loop.

    error_results : dict {label: error_result}, each from compute_errors on a
        different store (store_base, store_100samples, store_highNoise, ...).
        Order is preserved; put the base first. Errors are RMSE (from t=1).

    Two subplots side by side (conditions[0] left, conditions[1] right). Within
    each: x = loops, grouped bars = one per variation, whiskers = IQR. The figure
    widens with the number of variations."""
    labels_var = list(error_results.keys())
    n_var = len(labels_var)
    loop_labels = [LOOP_LABEL[k] for k in LOOP_KEYS]

    metric = "RMSE"

    palette = px.colors.qualitative.Prism
    colors = {lab: palette[i % len(palette)] for i, lab in enumerate(labels_var)}

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[str(c) for c in conditions],
                        horizontal_spacing=0.08, shared_yaxes=True)

    for ci, cond in enumerate(conditions, start=1):
        for lab in labels_var:
            summ = error_results[lab]["summary"][cond]
            med = [summ[k]["median"] for k in LOOP_KEYS]
            q1 = [summ[k]["q1"] for k in LOOP_KEYS]
            q3 = [summ[k]["q3"] for k in LOOP_KEYS]
            fig.add_trace(go.Bar(
                name=lab, legendgroup=lab, showlegend=(ci == 1),
                x=loop_labels, y=med, marker_color=colors.get(lab),
                error_y=dict(type="data", symmetric=False,
                             array=[hi - m for hi, m in zip(q3, med)],
                             arrayminus=[m - lo for m, lo in zip(med, q1)],
                             thickness=1.2, width=4),
                customdata=np.column_stack([q1, q3]),
                hovertemplate=("Loop %{x}<br>" + f"{lab} ({cond})"
                               f"<br>median {metric} %{{y:.4f}}"
                               "<br>IQR [%{customdata[0]:.4f}, "
                               "%{customdata[1]:.4f}]<extra></extra>"),
            ), row=1, col=ci)

    width = int(2 * (270 + 60 * n_var) + 160)

    fig.update_layout(
        barmode="group", template="plotly_white",
        height=600, width=width, bargap=0.25, bargroupgap=0.10,
        legend=dict(font=dict(size=10)),
    )
    fig.update_xaxes(title_text="Loop", row=1, col=1)
    fig.update_xaxes(title_text="Loop", row=1, col=2)
    fig.update_yaxes(title_text=f"{metric} (median +/- IQR)",
                     row=1, col=1)
    fig.update_yaxes(rangemode="tozero")
    return fig

In [ ]:
fig_error_comparison = plot_error_comparison({
    "base: 3650 samples, exact causal graph, σ=0.01":         error_base,
    "1000 samples": error_1000samples,
    "100 samples":  error_100samples,
    "σ=0.05":   error_highNoise,
    "weak confounding":  error_confounding_rho025_phi07,
    "strong confounding":  error_confounding_rho08_phi07,
    "all lags = 3, except self-lags = 1": error_lags_all3_self1,
    "all lags = {1,2,3}": error_lags_uncertainty
})

fig_error_comparison.show()
# fig_error_comparison.write_image('figures/fig_error_comparison.svg')

## 4. Loop contributions along forward and backward sweeps

In [ ]:
def sweep_simulate(L_schedule_daily, x0, F, p, window_len=100, step=50,
                   noise_std=0.01, seed=0, dt=0.1, additive_noise=False):
    """Simulate the SDDE while L follows the daily schedule (slow sweep), then cut sliding windows
    (length window_len, every `step` days) and store each window's mean state and mean L."""
    sub = int(round(1.0 / dt))
    L_schedule_fine = np.repeat(L_schedule_daily, sub)          # piecewise-constant per day
    traj_fine = simulate_sdde(x0, p, len(L_schedule_fine), dt=dt,
                              L_mean=L_schedule_fine, F_mean=F,
                              noise_std=noise_std, seed=seed, additive_noise=additive_noise)
    traj     = traj_fine[::sub]                                 # daily, raw

    T_daily = len(traj)
    n = window_len
    starts = list(range(0, T_daily - n + 1, step))
    centers     = np.array([s + n // 2 for s in starts])
    mean_states = np.array([traj[s:s+n].mean(axis=0) for s in starts])
    mean_Ls     = np.array([L_schedule_daily[s:s+n].mean() for s in starts])
    return dict(traj=traj, L_schedule=np.asarray(L_schedule_daily),
                starts=starts, window_len=n, step=step, centers=centers,
                mean_states=mean_states, mean_Ls=mean_Ls, F=F)

def sweep_decompose_analytical(sweep, params, T_max=20):
    """Analytical decomposition at each window's mean state."""
    F = sweep['F']
    analytical_decomp = []
    for mean_state in sweep['mean_states']:
        # dt=1.0: the analytical decomposition's lag axis is in days
        dec = decomposition_analytical_lagged(mean_state, F, params, dt=0.1, T_horizon=T_max)
        analytical_decomp.append(dec)
    return analytical_decomp


In [ ]:
from joblib import Parallel, delayed

def sweep_decompose_causaleffect(sweep, T_max=20, n_jobs=-1, detrend_window=None):
    """Data-driven decomposition of each sliding window (parallel with joblib). Overrides the serial version above."""
    win_len = sweep['window_len']
    det_win = detrend_window if detrend_window is not None else win_len
    wins = [
        detrend_traj(sweep['traj'][s:s + win_len], method='linear', window=det_win)
        for s in sweep['starts']
    ]
    return Parallel(n_jobs=n_jobs)(
        delayed(causaleffect_decomposition)(w, tau_max=T_max)
        for w in wins)

In [ ]:
def build_trajectory_figure(sweep_fw, sweep_bw):

    """Plot A and M against the slowly changing L for the forward and backward sweeps."""
    L_fw, traj_fw = sweep_fw['L_schedule'], sweep_fw['traj']
    L_bw, traj_bw = sweep_bw['L_schedule'], sweep_bw['traj']

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Trajectory: forward (L ↑)', 'Trajectory: backward (L ↓)'],
        shared_xaxes=True,
        horizontal_spacing=0.12
    )

    for col, (L_arr, traj) in enumerate([(L_fw, traj_fw), (L_bw, traj_bw)], start=1):
        fig.add_trace(
            go.Scatter(x=L_arr, y=traj[:, iA], mode='lines', line=dict(color=C['A'], width=0.6), name='Algae', legendgroup='A', showlegend=(col == 1)), row=1, col=col)
        fig.add_trace(
            go.Scatter(x=L_arr, y=traj[:, iM], mode='lines', line=dict(color=C['M'], width=0.6), name='Macrophytes', legendgroup='M', showlegend=(col == 1)), row=1, col=col)

    fig.update_xaxes(title_text='L')
    fig.update_yaxes(title_text='biomass')
    fig.update_layout(height=330, width=1000, legend=dict(font=dict(size=10)), template='plotly_white')

    return fig

In [ ]:
window_length = 1000
step_size = 10

F_sweep = 0.01
T_max = 100
noise_std = 0.01

# Forward sweep
T_fw = 365*50
L_fw = np.linspace(0.0, 1.5, T_fw)
print('increase of L in one window:', 1.5 * window_length / T_fw)
print('increase of L in one step:', 1.5 * step_size / T_fw)

x0_clear_seed = find_equilibria_from_initial_conditions(L=0, F=F_sweep, p=params)[0]
sweep_fw = sweep_simulate(L_fw, x0_clear_seed, F_sweep, params,
                          window_len=window_length, step=step_size, noise_std=noise_std, seed=8, dt=0.1, additive_noise=False)

# Backward sweep
T_bw = 365*50
L_bw = np.linspace(1.5, 0.0, T_bw)
x0_turbid_seed = find_equilibria_from_initial_conditions(L=1.5, F=F_sweep, p=params)[1]
sweep_bw = sweep_simulate(L_bw, x0_turbid_seed, F_sweep, params,
                          window_len=window_length, step=step_size, noise_std=noise_std, seed=0, dt=0.1, additive_noise=False)

traj_fw, Lc_fw, ms_fw = sweep_fw['traj'], sweep_fw['mean_Ls'], sweep_fw['mean_states']
traj_bw, Lc_bw, ms_bw = sweep_bw['traj'], sweep_bw['mean_Ls'], sweep_bw['mean_states']

In [ ]:
fig_traj = build_trajectory_figure(sweep_fw, sweep_bw)
fig_traj.show()

In [ ]:
L_tipp_fw = 1.3
L_tipp_bw = 0.512

In [ ]:
# # Sweep analytical decomposition
# analytical_decomp_fw = sweep_decompose_analytical(sweep_fw, params, T_max=T_max)
# analytical_decomp_bw = sweep_decompose_analytical(sweep_bw, params, T_max=T_max)

In [ ]:
# # Sweep causal effect decomposition
# ce_decomp_fw = sweep_decompose_causaleffect(sweep_fw, T_max=T_max, detrend_window=window_length)
# ce_decomp_bw = sweep_decompose_causaleffect(sweep_bw, T_max=T_max, detrend_window=window_length)

These are saved results as it takes quite some time to run:

In [ ]:
# _analytical_keys = ['ts', 'A_total', 'A_direct', 'A_nutrient', 'A_trophic', 'A_mixed', 
#                     'A_nutrient',
#              'A_eq', 'J']
# _ce_keys = ['ts', 'A_total', 'A_direct', 'A_trophic', 'A_mixed', 
#             'A_nutrient',
#              ]

# _r = np.load('sweep_results_both__Tmax100days_wl1000_step10_F001_T365times50_noM.npz')
# traj_fw, Lc_fw, ms_fw = _r['traj_fw'], _r['Lc_fw'], _r['ms_fw']
# traj_bw, Lc_bw, ms_bw = _r['traj_bw'], _r['Lc_bw'], _r['ms_bw']

# n_fw, n_bw = len(Lc_fw), len(Lc_bw)
# analytical_decomp_fw = [{k: _r[f'dfw_{k}'][i] for k in _analytical_keys} for i in range(n_fw)]
# analytical_decomp_bw = [{k: _r[f'dbw_{k}'][i] for k in _analytical_keys} for i in range(n_bw)]
# ce_decomp_fw = [{k: _r[f'tfw_{k}'][i] for k in _ce_keys} for i in range(n_fw)]
# ce_decomp_bw = [{k: _r[f'tbw_{k}'][i] for k in _ce_keys} for i in range(n_bw)]

### Sweep heatmaps

In [ ]:
def signed_log(x, eps=1e-3):
    """Sign-preserving log10 scaling, to make small and large loop values visible on one colour scale."""
    x = np.asarray(x, float)
    return np.sign(x) * np.log10(1.0 + np.abs(x) / eps)

In [ ]:
def sweep_heatmap(Lc_fw, Lc_bw, analytical_decomp_fw=None, analytical_decomp_bw=None, ce_decomp_fw=None, ce_decomp_bw=None,
                  source='causaleffect', T_show=30, clip_quantile=0.99,
                  tipp_fw=None, tipp_bw=None, window_L=None, L_range=None, mechs=None, transform=lambda x: x):
    """Heatmaps (lag T vs mean L) of each loop contribution, forward vs backward sweep.
    source = 'causaleffect' (data-driven) or 'analytical'; tipp_* mark the tipping points,
    window_L shades the L range covered by one window."""
    if mechs is None:
        mechs = [('A_total', 'Total'), ('A_direct', 'Direct'), ('A_nutrient', 'Nutrient'), ('A_trophic', 'Trophic'), ('A_mixed', 'Mixed')
                 ]
    fw_list = ce_decomp_fw if source == 'causaleffect' else analytical_decomp_fw
    bw_list = ce_decomp_bw if source == 'causaleffect' else analytical_decomp_bw

    ts = next(d for d in fw_list if d is not None)['ts']
    keep = ts >= 0 if T_show is None else (ts >= 0) & (ts <= T_show)
    ts_keep = ts[keep]

    def stack(dec_list, key):
        M = np.full((len(ts), len(dec_list)), np.nan)
        for i, d in enumerate(dec_list):
            if d is not None:
                M[:, i] = d[key]
        return M[keep]

    n_rows, vsp = len(mechs), 0.06
    titles = [t for (_, name) in mechs
              for t in (f'{name} — forward (L↑)', f'{name} — backward (L↓)')]
    fig = make_subplots(rows=n_rows, cols=2, subplot_titles=titles,
                        shared_xaxes='columns', shared_yaxes='rows',
                        horizontal_spacing=0.07, vertical_spacing=vsp)

    sub_h = (1.0 - (n_rows - 1) * vsp) / n_rows


    for r, (key, name) in enumerate(mechs, start=1):
        fw_M, bw_M = stack(fw_list, key), stack(bw_list, key)
        fw_M, bw_M = transform(fw_M), transform(bw_M)
        allv = np.abs(np.concatenate([fw_M.ravel(), bw_M.ravel()]))
        allv = allv[np.isfinite(allv)]
        vmax = np.quantile(allv, clip_quantile) if len(allv) else 1.0
        vmax = vmax if vmax > 0 else 1.0
        y_center = 1 - (r - 0.5) * sub_h - (r - 1) * vsp
        for c, (M, Lc) in enumerate([(fw_M, Lc_fw), (bw_M, Lc_bw)], start=1):
            order = np.argsort(Lc)            # heatmaps need x ascending; the backward
            M, Lc = M[:, order], np.asarray(Lc, float)[order]
            if L_range is not None: 
                pad = np.full((M.shape[0], 1), np.nan)
                M = np.hstack([pad, M, pad])                     
                Lc = np.concatenate([[L_range[0]], Lc, [L_range[1]]])
            fig.add_trace(
                go.Heatmap(z=M, x=Lc, y=ts_keep, colorscale='RdBu_r',
                           zmin=-vmax, zmax=vmax, zmid=0, showscale=(c == 2),
                           colorbar=dict(title=name, len=sub_h * 0.9, y=y_center,
                                         yanchor='middle', x=1.02, thickness=10,
                                         tickfont=dict(size=8))),
                row=r, col=c)
        fig.update_yaxes(title_text='T (days)', row=r, col=1)
        fig.update_yaxes(range=[float(ts_keep.min()) - 0.5, float(ts_keep.max()) + 0.5], row=r, col=1)
        fig.update_yaxes(range=[float(ts_keep.min()) - 0.5, float(ts_keep.max()) + 0.5], row=r, col=2)
        fig.update_yaxes(title_text='T (days)', row=r, col=1)


    y0b, y1b = float(ts_keep.min()) - 0.5, float(ts_keep.max()) + 0.5   # band spans the lag axis
    for col, L_t in [(1, tipp_fw), (2, tipp_bw)]:
        if L_t is None:
            continue
        for r in range(1, n_rows + 1):
            if window_L:
                x0b, x1b = L_t - window_L / 2, L_t + window_L / 2
                fig.add_trace(
                    go.Scatter(x=[x0b, x1b, x1b, x0b, x0b], y=[y0b, y0b, y1b, y1b, y0b],
                               mode='lines', line=dict(width=0), fillcolor='rgba(0,0,0,0)',
                               fill='toself', fillpattern=dict(shape='/', fgcolor='grey',
                                   bgcolor='rgba(0,0,0,0)', size=8, solidity=0.3),
                               showlegend=False, hoverinfo='skip'),
                    row=r, col=col)
            fig.add_vline(x=L_t, line=dict(color='grey', width=1.2, dash='dot'),
                          row=r, col=col)

    fig.update_xaxes(title_text='L<sub>c</sub>', row=n_rows, col=1)
    fig.update_xaxes(title_text='L<sub>c</sub>', row=n_rows, col=2)
    for ann in fig.layout.annotations:
        ann.font = dict(size=11)
    fig.update_layout(height=180 * n_rows, width=1000, template='plotly_white',
                      margin=dict(r=110, t=70),)
    return fig

In [ ]:
# window_length = 1000
window_L = window_length * abs(L_fw[-1] - L_fw[0]) / (len(L_fw) - 1)

fig_sweep_heatmaps_datadriven = sweep_heatmap(Lc_fw=Lc_fw, Lc_bw=Lc_bw, 
                                              ce_decomp_fw=ce_decomp_fw, ce_decomp_bw=ce_decomp_bw, 
                                              source='causaleffect', T_show=150, clip_quantile=0.98, 
                                              tipp_fw=L_tipp_fw, tipp_bw=L_tipp_bw, window_L=window_L, 
                                              L_range=(float(min(L_fw.min(), L_bw.min())), float(max(L_fw.max(), L_bw.max()))),
                                              transform=lambda x: signed_log(x, eps=1e-3))
fig_sweep_heatmaps_datadriven.show()

In [ ]:
window_L = window_length * abs(L_fw[-1] - L_fw[0]) / (len(L_fw) - 1)

fig_sweep_heatmaps_analytical = sweep_heatmap(Lc_fw=Lc_fw, Lc_bw=Lc_bw,
                                              analytical_decomp_fw=analytical_decomp_fw, analytical_decomp_bw=analytical_decomp_bw,
                                              source='analytical', T_show=100, clip_quantile=0.98,
                                              tipp_fw=L_tipp_fw, tipp_bw=L_tipp_bw, window_L=window_L,
                                              L_range=(float(min(L_fw.min(), L_bw.min())), float(max(L_fw.max(), L_bw.max()))),
                                              transform=lambda x: signed_log(x, eps=1e-3))
fig_sweep_heatmaps_analytical.show()